In [4]:
import requests
import pandas as pd
from io import StringIO

dataset_ids = ["E-MTAB-13553", "E-MTAB-12584", "E-MTAB-12916", "E-MTAB-12975", "E-MTAB-11030"]

# ----------------------------
# SDRF Parser
# ----------------------------
def extract_sdrf(ds_id):
    """Fetch SDRF and extract metadata"""
    url = f"https://www.ebi.ac.uk/arrayexpress/files/{ds_id}/{ds_id}.sdrf.txt"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        return None
    
    df_sdrf = pd.read_csv(StringIO(r.text), sep="\t")
    sample_size = df_sdrf.shape[0]

    def unique_vals(df, col_key):
        cols = df.filter(like=col_key)
        if cols.shape[1] == 0:
            return "Not Available"
        vals = cols.values.flatten()
        vals = pd.Series(vals).dropna().unique().tolist()
        return "; ".join([str(v) for v in vals]) if vals else "Not Available"

    organism = unique_vals(df_sdrf, "Characteristics[organism]")
    tissue   = unique_vals(df_sdrf, "Characteristics[organism part]")
    cell     = unique_vals(df_sdrf, "Characteristics[cell type]")
    omics    = unique_vals(df_sdrf, "Comment[LIBRARY_SOURCE]")

    return {
        "Sample Size": sample_size,
        "Organism": organism,
        "Tissue / Cell Type": tissue if tissue != "Not Available" else cell,
        "Omics Type": omics
    }

# ----------------------------
# IDF Parser (for title & pubs)
# ----------------------------
def extract_idf_info(ds_id):
    """Fetch IDF and extract study title & publication info"""
    url = f"https://www.ebi.ac.uk/arrayexpress/files/{ds_id}/{ds_id}.idf.txt"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        return {"title": "Not Available", "pub": "Not Available"}
    
    title, pub = "Not Available", "Not Available"
    for line in r.text.splitlines():
        if line.startswith("Investigation Title") or line.startswith("Comment[Submitted Name]"):
            parts = line.split("\t")
            if len(parts) > 1:
                title = parts[1].strip()
        if line.startswith("PubMed ID") or line.startswith("Comment[Publication Title]"):
            parts = line.split("\t")
            if len(parts) > 1:
                pub = parts[1].strip()
    return {"title": title, "pub": pub}

# ----------------------------
# Ontology Maps
# ----------------------------
ncbi_map = {
    "Homo sapiens": "NCBITaxon:9606",
    "Mus musculus": "NCBITaxon:10090",
    "Rattus norvegicus": "NCBITaxon:10116"
}

omics_map = {
    "transcriptomic": "RNA-Seq",
    "transcriptomic single cell": "Single-Cell RNA-Seq",
    "rna-seq": "RNA-Seq",
    "scrna-seq": "Single-Cell RNA-Seq",
    "proteomics": "Proteomics"
}

uberon_map = {
    "heart left ventricle": "UBERON:0002084",
    "heart right ventricle": "UBERON:0002082",
    "cardiac muscle cell": "CL:0000746",
    "right cardiac atrium": "UBERON:0002079",
    "apical region of left ventricle": "UBERON:0002084",
    "left cardiac atrium": "UBERON:0002079",
    "interventricular septum": "UBERON:0002094",
    "sinoatrial node": "UBERON:0002351",
    "atrioventricular node": "UBERON:0002087",
    "atrioventriculeft cardiac atriumr node": "UBERON:0002087"
}

# ----------------------------
# Main Loop
# ----------------------------
records = []

for ds_id in dataset_ids:
    # Try SDRF
    meta = extract_sdrf(ds_id)

    # API metadata
    api_url = f"https://www.ebi.ac.uk/biostudies/api/v1/studies/{ds_id}"
    r = requests.get(api_url)
    title, pub_year, doi = "Not Available", "Not Available", "Not Available"

    if r.status_code == 200:
        data = r.json()
        
        # Title from API
        title = data.get("title", "Not Available")
        if (not title or title == "Not Available") and "attributes" in data:
            for attr in data["attributes"]:
                if attr.get("name", "").lower() == "title":
                    title = attr.get("value", "Not Available")
                    break
        
        # Publications
        for pub in data.get("publications", []):
            doi = pub.get("doi", doi)
            pub_year = pub.get("year", pub_year)

    # Fallback to IDF if still missing
    if not title or title == "Not Available":
        idf_info = extract_idf_info(ds_id)
        title = idf_info["title"]
        if doi == "Not Available" and idf_info["pub"] != "Not Available":
            doi = idf_info["pub"]

    # If SDRF missing, fill blanks
    if meta is None:
        meta = {"Sample Size": "Not Available", "Organism": "Not Available",
                "Tissue / Cell Type": "Not Available", "Omics Type": "Not Available"}

    # Add record
    records.append({
        "Dataset ID": ds_id,
        "Study Title": title,
        "Organism": meta["Organism"],
        "Omics Type": meta["Omics Type"],
        "Sample Size": meta["Sample Size"],
        "Tissue / Cell Type": meta["Tissue / Cell Type"],
        "Publication Year": pub_year, ------#If missing, can use AI-assistance + Manual Validation
        "DOI / Publication Link": doi ------#If missing, can use AI-assistance + Manual Validation
    })

# ----------------------------
# Harmonization
# ----------------------------
df = pd.DataFrame(records)

# Organism
df["Organism"] = df["Organism"].map(ncbi_map).fillna(df["Organism"])

# Omics type (normalize to lowercase before mapping)
df["Omics Type"] = df["Omics Type"].str.lower()
df["Omics Type"] = df["Omics Type"].map(omics_map).fillna(df["Omics Type"])

# Tissue harmonization
def map_tissue(t):
    if pd.isna(t):
        return t
    return "; ".join([uberon_map.get(x.strip().lower(), x.strip()) for x in t.split(";")])

df["Tissue / Cell Type"] = df["Tissue / Cell Type"].apply(map_tissue)

# ----------------------------
# Save
# ----------------------------
output_path = r"C:\Users\kiran\OneDrive\Desktop\Jesus_Elucidata\ArrayExpress_metadata_curated.csv"
df.to_csv(output_path, index=False)
print("✅ ArrayExpress_metadata_curated.csv saved to:", output_path)

df


✅ ArrayExpress_metadata_curated.csv saved to: C:\Users\kiran\OneDrive\Desktop\Jesus_Elucidata\ArrayExpress_metadata_curated.csv


,Dataset ID,Study Title,Organism,Omics Type,Sample Size,Tissue / Cell Type,Publication Year,DOI / Publication Link
0,E-MTAB-13553,Cardiac biopsies reveal differences in transcr...,NCBITaxon:9606,RNA-Seq,41,UBERON:0002082; UBERON:0002084,Not Available,Not Available
1,E-MTAB-12584,Single nucleus RNA sequencing data from human ...,NCBITaxon:9606,Single-Cell RNA-Seq,138,UBERON:0002082; UBERON:0002084,Not Available,Not Available
2,E-MTAB-12916,Spatially resolved multiomics of human cardiac...,NCBITaxon:9606,Single-Cell RNA-Seq,188,UBERON:0002084; UBERON:0002079; UBERON:0002084...,Not Available,Not Available
3,E-MTAB-12975,Spatially resolved multiomics of human cardiac...,NCBITaxon:9606,RNA-Seq,168,UBERON:0002079; UBERON:0002079; UBERON:0002084...,Not Available,Not Available
4,E-MTAB-11030,A multi omics analysis of a human stem cell ba...,NCBITaxon:9606,RNA-Seq,160,CL:0000746,Not Available,Not Available
